# Watershed analytics

End-to-end demonstration of the four watershed-partitioning APIs and the W-7/8/9 metric
additions:

* `FlowDirection.watershed(points)` — pour-point delineation.
* `FlowDirection.basins()` — terminal-outlet partition.
* `FlowDirection.subbasins_pfafstetter(max_level=N)` — hierarchical coding.
* `FlowDirection.isobasins(streams, accumulation, target_area_km2)` — **W-7** equal-area split.
* `WatershedRaster.statistics(flow_direction=...)` — **W-8** longest_flow_path_m metric.
* `FlowDirection.upslope_flowpath_length()` — **W-9** per-cell upslope length raster.

In [ ]:
import numpy as np
import geopandas as gpd
from pyramids.dataset import Dataset
from shapely.geometry import Point
from digitalrivers import DEM

# Build a synthetic catchment: 25×25 grid, V-shaped main valley with two side tributaries.
rows, cols = 25, 25
z = np.full((rows, cols), 200.0, dtype=np.float32)
for r in range(rows):
    z[r, 12] = float(50 - 1.5 * r)  # main stem south
for c in range(12):
    z[8, c] = float(45 - c * 0.5)    # west tributary
for c in range(13, cols):
    z[17, c] = float(30 - (c - 13) * 0.5)

ds = Dataset.create_from_array(
    z, top_left_corner=(500_000.0, 5_000_000.0), cell_size=30.0,
    epsg=32618, no_data_value=-9999.0,
)
dem = DEM(ds.raster)
filled = dem.fill_depressions(method="priority_flood")
fd = filled.flow_direction(method="d8")
acc = fd.accumulate()
streams = acc.streams(threshold=3)
print(f"Stream cells: {int(streams.read_array().astype(bool).sum())}")

## 1. Pour-point watershed

Place a pour point and delineate everything upstream of it.

In [ ]:
gt = fd.geotransform
outlet_row, outlet_col = 24, 12  # south edge of the main stem
outlet_x = gt[0] + (outlet_col + 0.5) * gt[1]
outlet_y = gt[3] + (outlet_row + 0.5) * gt[5]
points = gpd.GeoDataFrame(
    {"id": [1]}, geometry=[Point(outlet_x, outlet_y)], crs=fd.epsg,
)
ws = fd.watershed(points)
print(f"Pour-point watershed cells: {int((ws.read_array() == 1).sum())}")

## 2. Terminal-outlet basins

Partition the whole DEM into basins keyed on terminal outlets.

In [ ]:
basins = fd.basins()
print(f"Number of basins: {basins.basin_count}")

## 3. Hierarchical Pfafstetter coding

Decimal-digit hierarchy: codes 1/3/5/7/9 = inter-basins; 2/4/6/8 = four largest tributaries
(ordered downstream-most-first).

In [ ]:
pfaf = fd.subbasins_pfafstetter(acc, streams, level=2)
codes = np.unique(pfaf.read_array())
codes = codes[codes != 0]
print(f"Pfafstetter codes (max 10 shown): {sorted(set(int(c) for c in codes))[:10]}")

## W-7 — Isobasin equal-area partition

Carve the catchment into sub-basins of approximately equal area. Useful for SWAT / HEC-HMS
modelling where each sub-basin must be ≤ a maximum unit.

In [ ]:
cell_area_km2 = abs(gt[1] * gt[5]) / 1.0e6
target = cell_area_km2 * 30  # target ~30 cells per sub-basin
iso = fd.isobasins(streams, acc, target_area_km2=target)
print(f"Target area:    {target * 1.0e6:.0f} m² (~{int(target / cell_area_km2)} cells)")
print(f"Sub-basin count: {iso.basin_count}")

## W-8 — Longest flow path per basin

Post-M1 fix: triggers on `flow_direction` alone. The per-basin metric is the longest
source-to-outlet path within that basin.

In [ ]:
stats = basins.statistics(dem=filled, flow_direction=fd, streams=streams)
print(stats[["area_km2", "mean_elev", "longest_flow_path_m", "drainage_density_km_per_km2"]])

## W-9 — Per-cell upslope flow-path length

Returns a raster of per-cell longest upslope flow path. Used as a building block for
time-of-concentration models, sensitivity analyses, and response-time mapping.

In [ ]:
upslope = fd.upslope_flowpath_length()
ups = upslope.read_array()
print(f"Upslope length range: [{ups.min():.1f}, {ups.max():.1f}] m")
print(f"Mean upslope length:  {ups[ups > 0].mean():.1f} m")

## Summary

Five distinct watershed partitions on the same DEM:
* Pour-point: 1 basin upstream of a chosen outlet.
* Terminal-outlet: every internal sink becomes its own basin.
* Pfafstetter: hierarchical decimal coding.
* Isobasin: approximately equal-area sub-basins (W-7).
* Plus per-basin `longest_flow_path_m` (W-8) and a per-cell upslope length raster (W-9).